# 믿:음 2.0 Base + Qwen3 음성 + MuseTalk 1.5 · Colab A100 통합 서버

이 노트북은 `K-intelligence/Midm-2.0-Base-Instruct` 4-bit, Qwen3-TTS/ASR와 MuseTalk 1.5를 **한 A100 런타임**에 로드하고, 가족센터 앱이 호출할 수 있는 LLM·음성·2D 립싱크 API를 하나의 HTTPS 주소로 엽니다.

- 시연·합성 페르소나 데이터만 사용하세요. 실제 내담자 개인정보는 보내지 마세요.
- Colab 탭과 런타임이 살아 있는 동안만 동작하며, 재연결하면 터널 URL이 바뀝니다.
- 먼저 `런타임 > 런타임 유형 변경 > A100 GPU`를 선택하세요. 이 통합 노트북은 T4/L4를 권장하지 않습니다.
- Colab 왼쪽의 **열쇠(Secrets)** 에 `NGROK_AUTHTOKEN`을 추가하고 노트북 액세스를 허용하세요. `HF_TOKEN`은 다운로드 제한이 생길 때만 선택적으로 추가합니다.
- ngrok 무료 계정과 authtoken은 https://dashboard.ngrok.com/get-started/your-authtoken 에서 준비합니다.


## 1. 패키지 설치
처음 한 번 수 분이 걸릴 수 있습니다. 설치 후 런타임을 재시작하라는 메시지가 나와도 우선 다음 셀을 실행해 보세요.


In [ ]:
%pip -q install -U "transformers==4.57.6" "accelerate==1.12.0" "bitsandbytes>=0.45,<1" "fastapi>=0.115,<1" "uvicorn[standard]>=0.34,<1" "ngrok>=1.4,<2" "pydantic>=2.10,<3"


## 2. GPU와 Secret 확인
`MIDM_API_KEY` Secret은 선택 사항입니다. 없으면 이 런타임 전용 키를 자동 생성해 마지막에 출력합니다.


In [ ]:
import os
# Reduce CUDA allocator fragmentation when Mi:dm, Qwen and MuseTalk share one GPU.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import secrets
import torch
from google.colab import userdata

MODEL_ID = "K-intelligence/Midm-2.0-Base-Instruct"
SERVER_PORT = 8000
MAX_INPUT_TOKENS = 6144
MAX_OUTPUT_TOKENS = 1600

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. 런타임 > 런타임 유형 변경에서 T4 GPU 이상을 선택하세요.")

def optional_secret(name: str) -> str:
    try:
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

NGROK_AUTHTOKEN = optional_secret("NGROK_AUTHTOKEN")
HF_TOKEN = optional_secret("HF_TOKEN") or None
MIDM_API_KEY = optional_secret("MIDM_API_KEY") or secrets.token_urlsafe(32)
if not NGROK_AUTHTOKEN:
    raise RuntimeError("Colab Secrets에 NGROK_AUTHTOKEN을 추가하고 노트북 액세스를 켜세요.")

gpu_name = torch.cuda.get_device_name(0)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"GPU: {gpu_name}")
print(f"4-bit 계산 dtype: {compute_dtype}")
print("Secret 확인 완료 (실제 값은 표시하지 않음)")


## 3. 믿:음 Base 4-bit 로드
최초 실행은 약 23GB 원본 가중치를 내려받고 양자화하며, 환경에 따라 5~20분 정도 걸릴 수 있습니다. 완료 메시지가 나올 때까지 기다리세요.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
print(f"모델 로드 완료 · GPU allocated {allocated:.1f}GB / reserved {reserved:.1f}GB")


## 4. OpenAI 호환 API 정의 및 로컬 서버 시작
동시 생성은 GPU 메모리 급증을 피하기 위해 1건씩 처리합니다. 입력은 6,144토큰, 출력은 1,600토큰으로 제한합니다.


In [ ]:
import asyncio
import threading
import time
import uuid
from typing import Literal

import uvicorn
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="Mi:dm 2.0 Base Colab Demo Server", version="0.1.0")
generation_lock = threading.Lock()

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=50000)

class ChatCompletionRequest(BaseModel):
    model: str = MODEL_ID
    messages: list[ChatMessage] = Field(min_length=1, max_length=50)
    max_tokens: int = Field(default=900, ge=1, le=MAX_OUTPUT_TOKENS)
    temperature: float = Field(default=0.35, ge=0.0, le=2.0)
    top_p: float = Field(default=0.9, gt=0.0, le=1.0)
    stream: bool = False

def require_api_key(authorization: str | None = Header(default=None)) -> None:
    expected = f"Bearer {MIDM_API_KEY}"
    if not authorization or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="Invalid API key")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "gpu": gpu_name}

@app.get("/v1/models", dependencies=[Depends(require_api_key)])
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "K-intelligence"}]}

def generate_sync(request: ChatCompletionRequest):
    messages = [item.model_dump() for item in request.messages]
    with generation_lock:
        batch = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        # Mi:dm tokenizer가 token_type_ids를 반환해도 모델에는 전달하지 않습니다.
        # 지원하는 텐서만 명시적으로 선택해 Transformers 버전 차이도 흡수합니다.
        batch = {key: value for key, value in batch.items() if key in {"input_ids", "attention_mask"}}
        input_tokens = int(batch["input_ids"].shape[-1])
        if input_tokens > MAX_INPUT_TOKENS:
            raise ValueError(f"입력이 {input_tokens}토큰입니다. {MAX_INPUT_TOKENS}토큰 이하로 줄이세요.")
        batch = {key: value.to(model.device) for key, value in batch.items()}
        generation_args = {
            "max_new_tokens": request.max_tokens,
            "do_sample": request.temperature > 0,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "use_cache": True,
        }
        if request.temperature > 0:
            generation_args.update(temperature=max(0.01, request.temperature), top_p=request.top_p)
        with torch.inference_mode():
            output = model.generate(**batch, **generation_args)
        generated = output[0, input_tokens:]
        text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        # greedy 생성이 첫 토큰에서 EOS로 끝나는 경우 한 번만 sampling으로 재시도합니다.
        if not text and not generation_args["do_sample"]:
            retry_args = {**generation_args, "do_sample": True, "temperature": 0.35, "top_p": 0.9}
            with torch.inference_mode():
                output = model.generate(**batch, **retry_args)
            generated = output[0, input_tokens:]
            text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        if not text:
            token_preview = generated[:16].detach().cpu().tolist()
            raise ValueError(f"모델이 빈 응답을 생성했습니다. 생성 토큰: {token_preview}")
        return text, input_tokens, int(generated.shape[-1])

@app.post("/v1/chat/completions", dependencies=[Depends(require_api_key)])
async def chat_completions(request: ChatCompletionRequest):
    if request.stream:
        raise HTTPException(status_code=400, detail="이 시연 서버는 stream=false만 지원합니다.")
    try:
        text, prompt_tokens, completion_tokens = await asyncio.to_thread(generate_sync, request)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": MODEL_ID,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens, "total_tokens": prompt_tokens + completion_tokens},
    }

if "uvicorn_server" in globals():
    uvicorn_server.should_exit = True
    if "server_thread" in globals() and server_thread.is_alive():
        server_thread.join(timeout=10)
    if "server_thread" in globals() and server_thread.is_alive():
        raise RuntimeError("이전 API 서버가 아직 종료되지 않았습니다. 5초 후 이 셀만 다시 실행하세요.")
uvicorn_server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=SERVER_PORT, log_level="warning"))
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()
deadline = time.time() + 15
while not uvicorn_server.started and server_thread.is_alive() and time.time() < deadline:
    time.sleep(0.1)
if not uvicorn_server.started:
    raise RuntimeError(f"로컬 API가 포트 {SERVER_PORT}에서 시작되지 않았습니다. 서버 셀 출력을 확인하세요.")
print(f"로컬 API 시작: http://127.0.0.1:{SERVER_PORT} · 최신 서버 코드 적용됨")


## 5. 로컬 API 1차 테스트
첫 생성은 CUDA 준비 때문에 이후 요청보다 느릴 수 있습니다. 응답 내용이 출력되면 모델과 API가 정상입니다.


In [ ]:
import requests

headers = {"Authorization": f"Bearer {MIDM_API_KEY}", "Content-Type": "application/json"}
test_payload = {
    "model": MODEL_ID,
    "messages": [
        {"role": "system", "content": "너는 한국어로 간결하게 답하는 테스트 도우미다."},
        {"role": "user", "content": "연결 확인이라고 짧게 답해줘."},
    ],
    "max_tokens": 48,
    "temperature": 0.0,
    "stream": False,
}
response = requests.post(f"http://127.0.0.1:{SERVER_PORT}/v1/chat/completions", headers=headers, json=test_payload, timeout=180)
if not response.ok:
    try:
        error_body = response.json()
    except ValueError:
        error_body = response.text
    raise RuntimeError(
        f"믿:음 로컬 API HTTP {response.status_code}: {error_body}\n"
        "위의 FastAPI 서버 셀을 다시 실행한 뒤 이 테스트 셀을 재실행하세요."
    )
print(response.json()["choices"][0]["message"]["content"])


## 6. ngrok 임시 HTTPS 주소 열기
셀 출력의 `.env` 블록을 로컬 프로젝트 루트의 `.env`에 그대로 반영합니다. API 키가 있으므로 URL만 알아서는 생성 API를 호출할 수 없습니다.


In [ ]:
import inspect
import re
import requests
import ngrok

# 같은 Colab 런타임에서 이 셀을 다시 실행해도 리스너가 중복되지 않게 정리합니다.
try:
    listeners_result = ngrok.get_listeners()
    listeners = await listeners_result if inspect.isawaitable(listeners_result) else listeners_result
    for listener in listeners:
        close_result = listener.close()
        if inspect.isawaitable(close_result):
            await close_result
except Exception as cleanup_error:
    print(f"현재 런타임 리스너 정리 건너뜀: {cleanup_error}")

try:
    forward_result = ngrok.forward(SERVER_PORT, authtoken=NGROK_AUTHTOKEN)
    ngrok_listener = await forward_result if inspect.isawaitable(forward_result) else forward_result
    PUBLIC_URL = ngrok_listener.url().rstrip("/")
except ValueError as exc:
    error_text = " ".join(str(item) for item in exc.args)
    url_match = re.search(r"https://[A-Za-z0-9.-]+\.ngrok-free\.dev", error_text)
    if "ERR_NGROK_334" not in error_text or not url_match:
        raise
    candidate_url = url_match.group(0).rstrip("/")
    try:
        health_response = requests.get(
            f"{candidate_url}/health",
            headers={"ngrok-skip-browser-warning": "1"},
            timeout=15,
        )
        health_body = health_response.json() if health_response.ok else {}
    except Exception:
        health_response, health_body = None, {}
    if health_response is not None and health_response.ok and health_body.get("status") == "ok":
        PUBLIC_URL = candidate_url
        ngrok_listener = None
        print(f"이미 실행 중인 ngrok 주소를 재사용합니다: {PUBLIC_URL}")
    else:
        raise RuntimeError(
            f"이전 ngrok 주소가 계정에 남아 있지만 응답하지 않습니다: {candidate_url}\n"
            "이전 Colab 런타임을 종료하거나 ngrok 대시보드의 Endpoints에서 해당 주소를 중지한 뒤 이 셀을 다시 실행하세요."
        ) from exc
OPENAI_BASE_URL = f"{PUBLIC_URL}/v1"

print("\n===== 로컬 프로젝트 .env에 넣을 값 =====")
print("AI_PROVIDER=internal_openai")
print(f"INTERNAL_LLM_BASE_URL={OPENAI_BASE_URL}")
print(f"INTERNAL_LLM_MODEL={MODEL_ID}")
print(f"INTERNAL_LLM_API_KEY={MIDM_API_KEY}")
print("LLM_REQUEST_TIMEOUT=240")
print("LLM_HEALTH_TIMEOUT=12")
print("=========================================\n")
print("중요: Colab 런타임이 재연결되면 이 셀을 다시 실행하고 새 URL로 .env를 갱신하세요.")


## 7. 외부 주소 최종 테스트
여기까지 성공하면 Windows의 현재 FastAPI도 같은 주소를 호출할 수 있습니다.


In [ ]:
public_headers = {**headers, "ngrok-skip-browser-warning": "1"}
models_response = requests.get(f"{OPENAI_BASE_URL}/models", headers=public_headers, timeout=30)
models_response.raise_for_status()
print("외부 연결 정상:", models_response.json()["data"][0]["id"])


## 8. Windows 앱 연결
1. 위 출력값을 프로젝트 루트 `.env`에 붙여 넣습니다. 기존 키가 있으면 해당 줄을 교체합니다.
2. 실행 중인 **백엔드 PowerShell에서 `Ctrl+C`** 후 프로젝트 루트에서 다시 실행합니다.
```powershell
.\.venv\Scripts\python.exe -m pip install -r backend\requirements.txt
.\.venv\Scripts\python.exe -m uvicorn backend.app.main:app --host 127.0.0.1 --port 8100
```
3. 프론트엔드는 재시작하지 않아도 됩니다. `http://127.0.0.1:3000/training` 또는 상담 코파일럿 화면을 새로고침합니다.
4. 상단에 **믿:음 연결 정상**이 보이면 완료입니다. 오프라인이면 이 노트북의 6·7번 셀과 `.env` URL을 확인하세요.

Colab 탭을 닫거나 런타임이 종료되면 기존 화면은 유지되지만 새 AI 응답 생성은 실패합니다. 시연 시작 전에 7번 셀을 한 번 실행해 상태를 확인하세요.


## 9. MuseTalk 1.5 설치와 가중치 준비
위 1~8번에서 믿:음과 ngrok이 정상인 것을 확인한 뒤 실행합니다. **A100 런타임 전용**이며 최초 한 번 10~20분 정도 걸릴 수 있습니다. 현재 Colab의 PyTorch·Transformers는 유지하고 MuseTalk에 필요한 패키지만 추가합니다.


In [ ]:
import os, subprocess, sys, urllib.request
from importlib.metadata import version as package_version
from pathlib import Path

if "A100" not in torch.cuda.get_device_name(0).upper():
    raise RuntimeError(f"이 통합 노트북은 A100용입니다. 현재 GPU: {torch.cuda.get_device_name(0)}")

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
# 믿:음 로드 과정에서 이미 import된 huggingface_hub를 실행 중간에 업그레이드하면
# 메모리와 디스크의 모듈 버전이 섞입니다. 현재 정상 버전을 그대로 고정합니다.
hf_hub_version = package_version("huggingface-hub")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed",
    "diffusers==0.30.2", "opencv-python-headless>=4.9", "soundfile>=0.12",
    "librosa>=0.11", "einops>=0.8", "gdown>=5.2", "imageio[ffmpeg]>=2.34",
    "omegaconf>=2.3", "ffmpeg-python>=0.2", "moviepy>=2", "gTTS>=2.5",
    f"huggingface-hub=={hf_hub_version}"
], check=True)
subprocess.run([sys.executable, "-c",
    "import diffusers, huggingface_hub; print('dependency check:', diffusers.__version__, huggingface_hub.__version__)"
], check=True)

MUSETALK_ROOT = Path("/content/MuseTalk")
if not (MUSETALK_ROOT / "scripts" / "realtime_inference.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/TMElyralab/MuseTalk.git", str(MUSETALK_ROOT)], check=True)

from huggingface_hub import snapshot_download
snapshot_download("TMElyralab/MuseTalk", local_dir=MUSETALK_ROOT / "models", allow_patterns=["musetalkV15/*"])
snapshot_download("stabilityai/sd-vae-ft-mse", local_dir=MUSETALK_ROOT / "models/sd-vae", allow_patterns=["config.json", "diffusion_pytorch_model.safetensors", "diffusion_pytorch_model.bin"])
snapshot_download("openai/whisper-tiny", local_dir=MUSETALK_ROOT / "models/whisper", allow_patterns=["config.json", "pytorch_model.bin", "preprocessor_config.json"])

face_parse_dir = MUSETALK_ROOT / "models/face-parse-bisent"
face_parse_dir.mkdir(parents=True, exist_ok=True)
if not (face_parse_dir / "79999_iter.pth").exists():
    import gdown
    gdown.download(id="154JgKpzCPW82qINcVieuPH3fZ2e0P812", output=str(face_parse_dir / "79999_iter.pth"), quiet=False)
if not (face_parse_dir / "resnet18-5c106cde.pth").exists():
    urllib.request.urlretrieve("https://download.pytorch.org/models/resnet18-5c106cde.pth", face_parse_dir / "resnet18-5c106cde.pth")
print("MuseTalk 1.5 코드와 가중치 준비 완료")


## 10. A100용 MuseTalk 모델 상주 로드
공식 MuseTalk 1.5의 립싱크 모델은 그대로 사용합니다. 고정 정면 사진에 맞춰 무거운 MMPose 의존성만 얼굴 검출 기반 전처리로 교체해 Colab Python 버전 충돌을 줄였습니다. 모델은 한 번만 GPU에 올리고 감정별 사진 전처리를 캐시합니다.


In [ ]:
import importlib, shutil
from types import SimpleNamespace

preprocessing_path = MUSETALK_ROOT / "musetalk/utils/preprocessing.py"
preprocessing_path.write_text('''import os
import cv2
import numpy as np
import torch
from tqdm import tqdm
from face_detection import FaceAlignment, LandmarksType

device = "cuda" if torch.cuda.is_available() else "cpu"
fa = FaceAlignment(LandmarksType._2D, flip_input=False, device=device)
coord_placeholder = (0.0, 0.0, 0.0, 0.0)

def read_imgs(img_list):
    frames = []
    for img_path in tqdm(img_list, desc="portrait frames"):
        frame = cv2.imread(img_path)
        if frame is None:
            raise ValueError(f"이미지를 읽을 수 없습니다: {img_path}")
        frames.append(frame)
    return frames

def get_landmark_and_bbox(img_list, upperbondrange=0):
    frames = read_imgs(img_list)
    detections = fa.get_detections_for_batch(np.asarray(frames))
    coords = []
    for detected, frame in zip(detections, frames):
        if detected is None:
            coords.append(coord_placeholder)
            continue
        x1, y1, x2, y2 = map(int, detected)
        face_w, face_h = x2 - x1, y2 - y1
        frame_h, frame_w = frame.shape[:2]
        x_pad = int(face_w * 0.03)
        crop = [
            max(0, x1 - x_pad),
            max(0, y1 + int(face_h * 0.10) + int(upperbondrange)),
            min(frame_w, x2 + x_pad),
            min(frame_h, y2 + int(face_h * 0.05)),
        ]
        if crop[2] <= crop[0] or crop[3] <= crop[1]:
            coords.append(coord_placeholder)
        else:
            coords.append(crop)
    return coords, frames

def get_bbox_range(img_list, upperbondrange=0):
    coords, _ = get_landmark_and_bbox(img_list, upperbondrange)
    return f"고정 정면 사진 {len(coords)}장 · bbox shift {upperbondrange}"
''', encoding='utf-8')

os.chdir(MUSETALK_ROOT)
for value in (str(MUSETALK_ROOT), str(MUSETALK_ROOT / "musetalk/utils")):
    if value not in sys.path:
        sys.path.insert(0, value)
import scripts.realtime_inference as rt

rt.args = SimpleNamespace(
    version="v15", extra_margin=10, parsing_mode="jaw", skip_save_images=False,
    audio_padding_length_left=2, audio_padding_length_right=2,
)
rt.device = torch.device("cuda:0")
rt.vae, rt.unet, rt.pe = rt.load_all_model(
    unet_model_path="./models/musetalkV15/unet.pth",
    vae_type="sd-vae",
    unet_config="./models/musetalkV15/musetalk.json",
    device=rt.device,
)
rt.timesteps = torch.tensor([0], device=rt.device)
rt.pe = rt.pe.half().to(rt.device)
rt.vae.vae = rt.vae.vae.half().to(rt.device)
rt.unet.model = rt.unet.model.half().to(rt.device)
rt.audio_processor = rt.AudioProcessor(feature_extractor_path="./models/whisper")
rt.weight_dtype = rt.unet.model.dtype
rt.whisper = rt.WhisperModel.from_pretrained("./models/whisper").to(device=rt.device, dtype=rt.weight_dtype).eval()
rt.whisper.requires_grad_(False)
# MuseTalk의 공식 face-parsing 체크포인트는 legacy tar 형식입니다.
# PyTorch 2.6+의 바뀐 기본값과 충돌하므로, 공식 체크포인트를 읽는 이 생성자 안에서만
# weights_only=False를 적용하고 즉시 원래 torch.load로 복구합니다.
_original_torch_load = torch.load
def _load_trusted_musetalk_checkpoint(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)
torch.load = _load_trusted_musetalk_checkpoint
try:
    rt.fp = rt.FaceParsing(left_cheek_width=90, right_cheek_width=90)
finally:
    torch.load = _original_torch_load
avatar_cache = {}
avatar_lock = threading.Lock()
print(f"MuseTalk 1.5 GPU 상주 완료 · allocated {torch.cuda.memory_allocated(0)/1024**3:.1f}GB")


## 11. 2D 립싱크 API 등록
백엔드가 보내는 감정별 가상 인물 사진과 음성을 받아 MP4를 생성합니다. 내부 TTS 음성이 없을 때만 시연용 Google TTS를 사용합니다. 실제 내부망에서는 `INTERNAL_TTS_URL`을 연결하세요.


In [ ]:
import base64, gc, hashlib, math, mimetypes, re, wave
import cv2, numpy as np
from gtts import gTTS
from fastapi.responses import FileResponse

# The generated portrait clip contains 60 full-resolution frames. Passing all of
# them to FaceAlignment at once causes a ~13GB CUDA allocation even when the
# MuseTalk inference batch is 1. Detect faces in bounded chunks instead.
import musetalk.utils.preprocessing as avatar_preprocessing
AVATAR_FACE_DETECTION_BATCH_SIZE = 1

def get_landmark_and_bbox_batched(img_list, upperbondrange=0):
    frames = avatar_preprocessing.read_imgs(img_list)
    detections = []
    for start in range(0, len(frames), AVATAR_FACE_DETECTION_BATCH_SIZE):
        batch = np.asarray(frames[start:start + AVATAR_FACE_DETECTION_BATCH_SIZE])
        detections.extend(avatar_preprocessing.fa.get_detections_for_batch(batch))
    coords = []
    for detected, frame in zip(detections, frames):
        if detected is None:
            coords.append(avatar_preprocessing.coord_placeholder)
            continue
        x1, y1, x2, y2 = map(int, detected)
        face_w, face_h = x2 - x1, y2 - y1
        frame_h, frame_w = frame.shape[:2]
        x_pad = int(face_w * 0.03)
        crop = [
            max(0, x1 - x_pad),
            max(0, y1 + int(face_h * 0.10) + int(upperbondrange)),
            min(frame_w, x2 + x_pad),
            min(frame_h, y2 + int(face_h * 0.05)),
        ]
        coords.append(crop if crop[2] > crop[0] and crop[3] > crop[1] else avatar_preprocessing.coord_placeholder)
    gc.collect()
    torch.cuda.empty_cache()
    return coords, frames

avatar_preprocessing.get_landmark_and_bbox = get_landmark_and_bbox_batched
rt.get_landmark_and_bbox = get_landmark_and_bbox_batched

AVATAR_WORK = MUSETALK_ROOT / "family_avatar_work"
AVATAR_WORK.mkdir(exist_ok=True)
AVATAR_BATCH_SIZE = 1  # Official low-memory default; shared A100 also hosts Mi:dm and Qwen.
AVATAR_AUDIO_LEAD_MS = 160  # Measured mouth-motion lead; delay only the model-driving audio.
AVATAR_FRAME_SIZE = (1280, 720)  # Keep H.264/yuv420p dimensions even for every persona.

# Safe cell rerun: replace old handlers and discard Avatar objects created with the old batch size.
avatar_paths = {"/v1/avatar/status", "/v1/avatar/render", "/v1/avatar/media/{avatar_id}/{filename}"}
app.router.routes[:] = [route for route in app.router.routes if getattr(route, "path", None) not in avatar_paths]
avatar_cache.clear()
gc.collect()
torch.cuda.empty_cache()

class AvatarRenderRequest(BaseModel):
    turn_id: str = Field(min_length=3, max_length=100)
    text: str = Field(min_length=1, max_length=1200)
    emotion: Literal["neutral", "sad", "angry", "anxious", "hurt", "withdrawn"]
    persona_id: Literal["lee-jieun", "kim-minseok"] = "lee-jieun"
    source_image_base64: str = Field(min_length=100, max_length=12_000_000)
    source_image_mime_type: str = "image/png"
    audio_url: str | None = None
    cache_key: str | None = Field(default=None, max_length=120)

def prepare_audio(request: AvatarRenderRequest, job_dir: Path) -> Path:
    source = job_dir / "speech_input"
    if request.audio_url and request.audio_url.startswith("data:"):
        encoded = request.audio_url.split(",", 1)[1]
        source.write_bytes(base64.b64decode(encoded))
    elif request.audio_url and request.audio_url.startswith(("http://", "https://")):
        response = requests.get(request.audio_url, headers={"ngrok-skip-browser-warning": "1"}, timeout=120)
        response.raise_for_status()
        source.write_bytes(response.content)
    else:
        source = job_dir / "speech.mp3"
        gTTS(request.text, lang="ko").save(str(source))
    normalized = job_dir / "speech.wav"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(source), "-ar", "16000", "-ac", "1", str(normalized)], check=True)
    return normalized

def prepare_lipsync_audio(audio_path: Path, job_dir: Path) -> Path:
    # MuseTalk mouth motion measured about 160ms ahead of the final audio. Feed a
    # delayed copy to the model, then mux the undelayed original into the MP4.
    delayed = job_dir / "speech_for_lipsync.wav"
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(audio_path),
        "-af", f"adelay={AVATAR_AUDIO_LEAD_MS}:all=1", str(delayed),
    ], check=True)
    return delayed

def mux_original_audio(video_path: Path, audio_path: Path) -> None:
    corrected = video_path.with_name(f".{video_path.stem}.sync.mp4")
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(video_path), "-i", str(audio_path),
        "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy", "-c:a", "aac",
        "-b:a", "192k", "-shortest", "-movflags", "+faststart", str(corrected),
    ], check=True)
    corrected.replace(video_path)

def speech_activity_by_frame(audio_path: Path, frame_count: int, fps: int = 25) -> np.ndarray:
    with wave.open(str(audio_path), "rb") as wav_file:
        sample_rate = wav_file.getframerate()
        channels = wav_file.getnchannels()
        samples = np.frombuffer(wav_file.readframes(wav_file.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0
    if channels > 1:
        samples = samples.reshape(-1, channels).mean(axis=1)
    rms = []
    for index in range(frame_count):
        start = max(0, int((index / fps - 0.02) * sample_rate))
        end = min(len(samples), int((index / fps + 0.06) * sample_rate))
        window = samples[start:end]
        rms.append(float(np.sqrt(np.mean(window * window))) if len(window) else 0.0)
    rms = np.asarray(rms, dtype=np.float32)
    peak = float(np.percentile(rms, 95)) if len(rms) else 0.0
    noise = float(np.percentile(rms, 20)) if len(rms) else 0.0
    threshold = max(0.004, noise * 3.0, peak * 0.06)
    active = rms >= threshold
    # Keep quiet consonants and short natural pauses from flickering open/closed.
    return np.convolve(active.astype(np.uint8), np.ones(7, dtype=np.uint8), mode="same") > 0

def apply_silence_gate(video_path: Path, audio_path: Path, avatar, fps: int = 25) -> None:
    capture = cv2.VideoCapture(str(video_path))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if width <= 0 or height <= 0 or frame_count <= 0:
        capture.release()
        raise RuntimeError("MuseTalk 무음 프레임 보정용 영상을 읽지 못했습니다.")
    activity = speech_activity_by_frame(audio_path, frame_count, fps)
    gated = video_path.with_name(f".{video_path.stem}.gated.mp4")
    encoder = subprocess.Popen([
        "ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24",
        "-s", f"{width}x{height}", "-r", str(fps), "-i", "-", "-an",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p", str(gated),
    ], stdin=subprocess.PIPE, stderr=subprocess.PIPE)
    blend = 0.0
    written = 0
    try:
        while written < frame_count:
            ok, generated = capture.read()
            if not ok:
                break
            target = 1.0 if activity[written] else 0.0
            blend = min(1.0, blend + 0.50) if target else max(0.0, blend - 0.25)
            if blend < 1.0:
                resting = avatar.frame_list_cycle[written % len(avatar.frame_list_cycle)]
                if resting.shape[:2] != generated.shape[:2]:
                    resting = cv2.resize(resting, (width, height), interpolation=cv2.INTER_LANCZOS4)
                generated = cv2.addWeighted(generated, blend, resting, 1.0 - blend, 0.0)
            encoder.stdin.write(generated.tobytes())
            written += 1
    finally:
        capture.release()
        if encoder.stdin:
            encoder.stdin.close()
    stderr = encoder.stderr.read().decode("utf-8", errors="replace") if encoder.stderr else ""
    return_code = encoder.wait()
    if return_code != 0 or written == 0:
        raise RuntimeError(f"무음 입모양 보정 실패: {stderr[-600:]}")
    gated.replace(video_path)

def build_motion_frames(image: np.ndarray, input_dir: Path, frame_count: int = 60):
    # 고정 사진 전체를 크게 흔들지 않고, 2~3px 이내의 호흡·고개 미세 이동만 만듭니다.
    if (image.shape[1], image.shape[0]) != AVATAR_FRAME_SIZE:
        image = cv2.resize(image, AVATAR_FRAME_SIZE, interpolation=cv2.INTER_AREA)
    height, width = image.shape[:2]
    center = (width / 2, height / 2)
    for index in range(frame_count):
        phase = 2 * math.pi * index / frame_count
        scale = 1.0018 + 0.0015 * math.sin(phase)
        dx = 0.75 * math.sin(phase * 0.63)
        dy = 0.55 * math.sin(phase + 0.8)
        matrix = cv2.getRotationMatrix2D(center, 0.045 * math.sin(phase * 0.47), scale)
        matrix[0, 2] += dx
        matrix[1, 2] += dy
        frame = cv2.warpAffine(image, matrix, (width, height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT_101)
        cv2.imwrite(str(input_dir / f"{index:08d}.png"), frame)

def safe_cache_name(value: str | None):
    if not value:
        return None
    normalized = re.sub(r"[^A-Za-z0-9_-]", "", value)[:120]
    return normalized or None

def get_cached_avatar(request: AvatarRenderRequest):
    raw = base64.b64decode(request.source_image_base64)
    digest = hashlib.sha256(raw).hexdigest()[:16]
    avatar_id = f"{request.persona_id}-{request.emotion}-{digest}-motion3-720p"
    if avatar_id in avatar_cache:
        return avatar_id, avatar_cache[avatar_id]
    input_dir = AVATAR_WORK / avatar_id
    input_dir.mkdir(parents=True, exist_ok=True)
    decoded = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if decoded is None:
        raise ValueError("가상 인물 사진을 디코딩하지 못했습니다.")
    if not any(input_dir.glob("*.png")):
        build_motion_frames(decoded, input_dir)
    saved = MUSETALK_ROOT / "results/v15/avatars" / avatar_id
    required_cache = [saved / "latents.pt", saved / "coords.pkl", saved / "mask_coords.pkl"]
    cache_ready = all(path.exists() for path in required_cache)
    if saved.exists() and not cache_ready:
        shutil.rmtree(saved)  # 실패 중 남은 불완전 캐시는 자동 복구
    avatar = rt.Avatar(avatar_id=avatar_id, video_path=str(input_dir), bbox_shift=0, batch_size=AVATAR_BATCH_SIZE, preparation=not cache_ready)
    avatar_cache[avatar_id] = avatar
    return avatar_id, avatar

def render_avatar_sync(request: AvatarRenderRequest):
    safe_turn = safe_cache_name(request.cache_key) or re.sub(r"[^A-Za-z0-9_-]", "", request.turn_id)[:80] or uuid.uuid4().hex
    raw = base64.b64decode(request.source_image_base64)
    digest = hashlib.sha256(raw).hexdigest()[:16]
    avatar_id = f"{request.persona_id}-{request.emotion}-{digest}-motion3-720p"
    output = MUSETALK_ROOT / "results/v15/avatars" / avatar_id / "vid_output" / f"{safe_turn}.mp4"
    if request.cache_key and output.exists():
        return avatar_id, safe_turn, 0, True
    job_dir = AVATAR_WORK / safe_turn
    if job_dir.exists():
        shutil.rmtree(job_dir)
    job_dir.mkdir(parents=True)
    started = time.perf_counter()
    audio_path = prepare_audio(request, job_dir)
    lipsync_audio_path = prepare_lipsync_audio(audio_path, job_dir)
    with avatar_lock, generation_lock:
        gc.collect()
        torch.cuda.empty_cache()
        avatar_id, avatar = get_cached_avatar(request)
        try:
            avatar.inference(str(lipsync_audio_path), safe_turn, 25, False)
        finally:
            torch.cuda.empty_cache()
    output = MUSETALK_ROOT / "results/v15/avatars" / avatar_id / "vid_output" / f"{safe_turn}.mp4"
    if not output.exists():
        raise RuntimeError("MuseTalk 결과 MP4가 생성되지 않았습니다.")
    apply_silence_gate(output, audio_path, avatar, 25)
    mux_original_audio(output, audio_path)
    shutil.rmtree(job_dir, ignore_errors=True)
    return avatar_id, safe_turn, round((time.perf_counter() - started) * 1000), False

@app.get("/v1/avatar/status", dependencies=[Depends(require_api_key)])
def avatar_status_api():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    return {"status": "ok", "provider": "musetalk_v15", "gpu": gpu_name, "batch_size": AVATAR_BATCH_SIZE, "face_detection_batch_size": AVATAR_FACE_DETECTION_BATCH_SIZE, "cached_avatars": len(avatar_cache), "gpu_free_gb": round(free_bytes / 1024**3, 2), "gpu_total_gb": round(total_bytes / 1024**3, 2)}

@app.post("/v1/avatar/render", dependencies=[Depends(require_api_key)])
async def render_avatar_api(request: AvatarRenderRequest):
    try:
        avatar_id, safe_turn, render_ms, cached = await asyncio.to_thread(render_avatar_sync, request)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"MuseTalk 생성 실패: {exc}") from exc
    return {"provider": "musetalk_v15_motion", "emotion": request.emotion, "video_url": f"/v1/avatar/media/{avatar_id}/{safe_turn}.mp4", "render_ms": render_ms, "cached": cached}

@app.get("/v1/avatar/media/{avatar_id}/{filename}", dependencies=[Depends(require_api_key)])
def avatar_media(avatar_id: str, filename: str):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", avatar_id) or not re.fullmatch(r"[A-Za-z0-9_-]+\.mp4", filename):
        raise HTTPException(status_code=404, detail="Not found")
    path = MUSETALK_ROOT / "results/v15/avatars" / avatar_id / "vid_output" / filename
    if not path.exists():
        raise HTTPException(status_code=404, detail="Not found")
    return FileResponse(path, media_type="video/mp4", headers={"Cache-Control": "private, max-age=3600"})

print("2D 립싱크 API 등록 완료:", f"{PUBLIC_URL}/v1/avatar/render")


## 12. Qwen3 음성 패키지 설치
상담사 음성 인식은 경량 `Qwen3-ASR-0.6B`, 페르소나 음성 합성은 `Qwen3-TTS-12Hz-1.7B-VoiceDesign`을 사용합니다. 여성은 30대 한국인 여성, 남성은 40대 한국인 남성의 자연스러운 상담실 대화 음색으로 각각 설계하며 정서는 앱이 전달한 강도를 0.25~0.78 범위로 제한합니다. A100에서 한 번만 실행하세요.


In [ ]:
# qwen-asr의 현재 공식 PyPI 버전은 0.0.6입니다. qwen-tts 0.1.1과
# Transformers 패치 버전 제약이 달라, 공통 런타임 4.57.6에서 두 패키지만
# --no-deps로 설치하고 필요한 의존성은 명시적으로 준비합니다.
import importlib.util, subprocess, sys
from importlib.metadata import version as package_version

speech_dependencies = [
    "nagisa==0.2.11", "soynlp==0.0.493", "qwen-omni-utils",
    "sox", "gradio", "flask", "pytz", "onnxruntime",
    "python-multipart>=0.0.20,<1", "soundfile>=0.13,<1",
]
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--upgrade-strategy", "only-if-needed", *speech_dependencies,
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U", "--no-deps",
    "qwen-asr==0.0.6", "qwen-tts==0.1.1",
], check=True)

importlib.invalidate_caches()
missing_modules = [name for name in ("qwen_asr", "qwen_tts") if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(
        f"Qwen 음성 패키지 설치 확인 실패: {', '.join(missing_modules)}. "
        "이 셀의 pip 오류를 확인한 뒤 셀을 다시 실행하세요."
    )
print(
    "Qwen 음성 패키지 준비 완료 ·",
    f"qwen-asr {package_version('qwen-asr')} ·",
    f"qwen-tts {package_version('qwen-tts')} ·",
    f"transformers {package_version('transformers')}",
)


## 13. Qwen3 TTS·ASR 모델 로드와 API 등록
두 모델은 GPU에 한 번만 상주합니다. 동일한 GPU에서 믿:음·MuseTalk와 동시에 추론하지 않도록 기존 잠금을 공유해 메모리 급증을 막습니다. 남녀 음색과 감정 지시는 백엔드가 페르소나 상태에 맞춰 자동 전달합니다.


In [ ]:
import gc, io, tempfile
import soundfile as sf
from fastapi import File, UploadFile
from fastapi.responses import Response
from qwen_asr import Qwen3ASRModel
from qwen_tts import Qwen3TTSModel

QWEN_TTS_MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
QWEN_ASR_MODEL_ID = "Qwen/Qwen3-ASR-0.6B"

if "qwen_tts_model" not in globals():
    qwen_tts_model = Qwen3TTSModel.from_pretrained(
        QWEN_TTS_MODEL_ID, device_map="cpu", dtype=torch.bfloat16
    )
if "qwen_asr_model" not in globals():
    qwen_asr_model = Qwen3ASRModel.from_pretrained(
        QWEN_ASR_MODEL_ID, device_map="cpu", dtype=torch.bfloat16,
        max_inference_batch_size=4, max_new_tokens=256,
    )

class SpeechSynthesisRequest(BaseModel):
    text: str = Field(min_length=1, max_length=1600)
    language: str = "Korean"
    persona_id: Literal["lee-jieun", "kim-minseok"] = "lee-jieun"
    gender: Literal["female", "male"] = "female"
    speaker: str | None = None
    voice_description: str = Field(default="", max_length=700)
    emotion: Literal["neutral", "sad", "angry", "anxious", "hurt", "withdrawn"] = "neutral"
    emotion_intensity: float = Field(default=.55, ge=0, le=1)
    instruct: str = Field(default="", max_length=500)
    cache_key: str | None = Field(default=None, max_length=120)
    format: Literal["wav"] = "wav"

VOICE_BY_GENDER = {"female": "이지은 음성", "male": "김민석 음성"}
VOICE_DESCRIPTION_BY_GENDER = {
    "female": "30대 중반 한국인 여성의 자연스러운 일상 대화 목소리. 맑지만 지나치게 높지 않은 음역, 부드러운 현실적 호흡, 보통 속도. 성우나 뉴스 낭독처럼 연기하지 않고 상담실에서 바로 앞 사람에게 조용히 말하는 느낌",
    "male": "40대 초반 한국인 남성의 자연스러운 일상 대화 목소리. 너무 굵거나 나이 들어 보이지 않는 편안한 중저음, 정확한 한국어 억양, 보통 속도. 성우나 뉴스 낭독처럼 과장하지 않고 상담실에서 바로 앞 사람에게 조용히 말하는 느낌",
}
FALLBACK_STYLE = {
    "neutral": "차분하고 자연스럽게",
    "sad": "슬픔이 은은히 묻어나되 울먹이지 않게",
    "angry": "답답함이 느껴지되 공격적이지 않게",
    "anxious": "불안이 약간 묻어나되 과장하지 않게",
    "hurt": "상처받은 마음이 조용히 느껴지게",
    "withdrawn": "담담하고 조심스럽게, 음량을 약간 낮춰",
}

QWEN_TTS_CACHE = Path("/content/qwen_tts_demo_cache")
QWEN_TTS_CACHE.mkdir(parents=True, exist_ok=True)

# This cell is safe to rerun in an active Colab runtime. Replace older speech routes
# instead of leaving duplicate FastAPI routes that would keep serving the old handler.
speech_paths = {"/v1/speech/status", "/v1/audio/speech", "/v1/audio/transcriptions"}
app.router.routes[:] = [route for route in app.router.routes if getattr(route, "path", None) not in speech_paths]

def place_qwen_model(wrapper, device: str):
    wrapper.model.to(device)
    wrapper.device = torch.device(device)
SPEECH_GPU_SCHEDULER_ACTIVE = True

def synthesize_qwen_sync(request: SpeechSynthesisRequest):
    speaker = VOICE_BY_GENDER[request.gender]
    cache_name = safe_cache_name(request.cache_key)
    cache_path = QWEN_TTS_CACHE / f"{cache_name}.wav" if cache_name else None
    if cache_path and cache_path.exists():
        return cache_path.read_bytes(), speaker, min(.78, max(.25, request.emotion_intensity)), True
    intensity = min(.78, max(.25, request.emotion_intensity))
    emotion_style = request.instruct.strip() or f"{FALLBACK_STYLE[request.emotion]} 말한다. 강도 {intensity:.2f}, 실제 상담처럼 절제한다."
    voice_description = request.voice_description.strip() or VOICE_DESCRIPTION_BY_GENDER[request.gender]
    instruct = f"{voice_description}. {emotion_style} 문장 사이의 호흡과 머뭇거림은 자연스럽게 두되, 과도한 감정 연기와 기계적인 낭독은 피한다."
    with generation_lock, torch.random.fork_rng(devices=[0]):
        place_qwen_model(qwen_tts_model, "cuda:0")
        try:
            seed = int(hashlib.sha256(request.persona_id.encode("utf-8")).hexdigest()[:8], 16)
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            wavs, sample_rate = qwen_tts_model.generate_voice_design(
                text=request.text, language="Korean", instruct=instruct
            )
        finally:
            place_qwen_model(qwen_tts_model, "cpu")
            gc.collect()
            torch.cuda.empty_cache()
    buffer = io.BytesIO()
    sf.write(buffer, wavs[0], sample_rate, format="WAV")
    raw = buffer.getvalue()
    if cache_path:
        cache_path.write_bytes(raw)
    del wavs
    gc.collect()
    torch.cuda.empty_cache()
    return raw, speaker, intensity, False

@app.get("/v1/speech/status", dependencies=[Depends(require_api_key)])
def qwen_speech_status():
    return {"status": "ok", "tts_model": QWEN_TTS_MODEL_ID, "asr_model": QWEN_ASR_MODEL_ID, "voices": VOICE_BY_GENDER}

@app.post("/v1/audio/speech", dependencies=[Depends(require_api_key)])
async def synthesize_qwen(request: SpeechSynthesisRequest):
    try:
        wav, speaker, intensity, cached = await asyncio.to_thread(synthesize_qwen_sync, request)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"음성 합성 실패: {exc}") from exc
    # HTTP response headers must stay ASCII/Latin-1 safe. Korean display labels such as
    # '이지은 음성' caused Starlette to raise UnicodeEncodeError after synthesis completed.
    return Response(wav, media_type="audio/wav", headers={"X-Qwen-Persona": request.persona_id, "X-Qwen-Gender": request.gender, "X-Emotion-Intensity": f"{intensity:.2f}", "X-Cache-Hit": str(cached).lower()})

def transcribe_qwen_sync(audio_path: str):
    with generation_lock:
        place_qwen_model(qwen_asr_model, "cuda:0")
        try:
            results = qwen_asr_model.transcribe(audio=audio_path, language="Korean")
        finally:
            place_qwen_model(qwen_asr_model, "cpu")
            gc.collect()
            torch.cuda.empty_cache()
    if not results or not results[0].text.strip():
        raise RuntimeError("인식 결과가 비어 있습니다.")
    return results[0].text.strip(), results[0].language

@app.post("/v1/audio/transcriptions", dependencies=[Depends(require_api_key)])
async def transcribe_qwen(file: UploadFile = File(...)):
    raw = await file.read()
    if not raw or len(raw) > 25 * 1024 * 1024:
        raise HTTPException(status_code=400, detail="음성 파일은 25MB 이하여야 합니다.")
    suffix = Path(file.filename or "speech.webm").suffix or ".webm"
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as temp:
            temp.write(raw)
            temp_path = temp.name
        text, language = await asyncio.to_thread(transcribe_qwen_sync, temp_path)
        return {"text": text, "language": language, "provider": "qwen3_asr"}
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"음성 인식 실패: {exc}") from exc
    finally:
        if temp_path and Path(temp_path).exists():
            Path(temp_path).unlink()

print("Qwen3 음성 API 등록 완료:", f"{PUBLIC_URL}/v1/audio/speech", f"{PUBLIC_URL}/v1/audio/transcriptions")


## 13-1. 실행 중인 Colab 립싱크 메모리 긴급 적용
이미 서버를 켠 뒤에도 전체 런타임 재시작 없이 MuseTalk를 저메모리 배치 1로 바꾸고 Qwen TTS·ASR을 필요할 때만 GPU에 올리는 메모리 스케줄러를 적용합니다. 립싱크가 정지 사진으로만 보이거나 CUDA OOM이 발생할 때 이 셀만 한 번 실행하세요.


In [ ]:
import gc, threading

# Existing handlers resolve these globals at call time, so no route restart is required.
if "_ORIGINAL_MUSETALK_AVATAR_CLASS" not in globals():
    _ORIGINAL_MUSETALK_AVATAR_CLASS = rt.Avatar

def _avatar_with_safe_batch(*args, **kwargs):
    kwargs["batch_size"] = 1
    return _ORIGINAL_MUSETALK_AVATAR_CLASS(*args, **kwargs)

rt.Avatar = _avatar_with_safe_batch
AVATAR_BATCH_SIZE = 1
generation_lock = threading.RLock()

def _place_qwen_for_hotfix(wrapper, device: str):
    wrapper.model.to(device)
    wrapper.device = torch.device(device)

# Older active notebooks keep both Qwen models on CUDA. Wrap their inference so
# cached demo speech stays instant and non-cached speech temporarily borrows the GPU.
if not globals().get("SPEECH_GPU_SCHEDULER_ACTIVE"):
    _BASE_SYNTHESIZE_QWEN_SYNC = synthesize_qwen_sync
    _BASE_TRANSCRIBE_QWEN_SYNC = transcribe_qwen_sync

    def synthesize_qwen_sync(request):
        cache_name = safe_cache_name(request.cache_key)
        cache_path = QWEN_TTS_CACHE / f"{cache_name}.wav" if cache_name else None
        if cache_path and cache_path.exists():
            return _BASE_SYNTHESIZE_QWEN_SYNC(request)
        with generation_lock:
            _place_qwen_for_hotfix(qwen_tts_model, "cuda:0")
            try:
                return _BASE_SYNTHESIZE_QWEN_SYNC(request)
            finally:
                _place_qwen_for_hotfix(qwen_tts_model, "cpu")
                gc.collect()
                torch.cuda.empty_cache()

    def transcribe_qwen_sync(audio_path: str):
        with generation_lock:
            _place_qwen_for_hotfix(qwen_asr_model, "cuda:0")
            try:
                return _BASE_TRANSCRIBE_QWEN_SYNC(audio_path)
            finally:
                _place_qwen_for_hotfix(qwen_asr_model, "cpu")
                gc.collect()
                torch.cuda.empty_cache()

    SPEECH_GPU_SCHEDULER_ACTIVE = True

# Free the resident Qwen weights now; cached female/male demo WAV files remain available.
_place_qwen_for_hotfix(qwen_tts_model, "cpu")
_place_qwen_for_hotfix(qwen_asr_model, "cpu")
avatar_cache.clear()
gc.collect()
torch.cuda.empty_cache()
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"MuseTalk 긴급 적용 완료 · batch={AVATAR_BATCH_SIZE} · GPU free {free_bytes/1024**3:.1f}/{total_bytes/1024**3:.1f}GB")


## 13-2. 시연용 남녀 립싱크 MP4 한 번만 굽기
13-1 실행 후에도 GPU free가 4GB 미만이면 이 셀을 실행합니다. 기본 질문의 남녀 WAV는 이미 로컬 백엔드에 저장되어 있으므로 현재 런타임에서 믿:음·Qwen 모델을 내리고 MuseTalk에 메모리를 집중합니다. 이후 교육 화면을 열면 남녀 MP4가 `backend/data/demo_media/`에 저장됩니다. 영상 생성 중에는 일반 AI 대화와 STT를 사용하지 마세요.


In [ ]:
import gc

# Release non-MuseTalk models only for the one-time demo video bake.
for model_name in ("model", "qwen_tts_model", "qwen_asr_model"):
    loaded_model = globals().pop(model_name, None)
    if loaded_model is not None:
        del loaded_model
gc.collect()
torch.cuda.empty_cache()
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"영상 굽기 모드 준비 완료 · MuseTalk batch={AVATAR_BATCH_SIZE} · GPU free {free_bytes/1024**3:.1f}/{total_bytes/1024**3:.1f}GB")
print("이제 로컬 교육 화면을 새로고침하고 '설정 적용·새 실습'을 누르세요.")


## 14. 로컬 `.env` 최종값
아래 블록 전체를 로컬 프로젝트 루트 `.env`에 반영한 뒤 **백엔드만 재시작**합니다. 모델 설치·로드 셀은 같은 런타임에서 반복 실행하지 마세요.


In [ ]:
print("\n===== backend .env configuration =====")
print("AI_PROVIDER=internal_openai")
print(f"INTERNAL_LLM_BASE_URL={OPENAI_BASE_URL}")
print(f"INTERNAL_LLM_MODEL={MODEL_ID}")
print(f"INTERNAL_LLM_API_KEY={MIDM_API_KEY}")
print("AVATAR_PROVIDER=internal_http")
print(f"INTERNAL_AVATAR_BASE_URL={PUBLIC_URL}")
print(f"INTERNAL_AVATAR_API_KEY={MIDM_API_KEY}")
print("AVATAR_REQUEST_TIMEOUT=300")
print(f"INTERNAL_TTS_URL={PUBLIC_URL}/v1/audio/speech")
print(f"INTERNAL_TTS_API_KEY={MIDM_API_KEY}")
print("TTS_REQUEST_TIMEOUT=180")
print("STT_PROVIDER=qwen_http")
print(f"INTERNAL_STT_URL={PUBLIC_URL}/v1/audio/transcriptions")
print(f"INTERNAL_STT_API_KEY={MIDM_API_KEY}")
print("STT_REQUEST_TIMEOUT=120")
print("STT_HEALTH_TIMEOUT=8")
print("\nA100 통합 서버 준비 완료. Colab 탭과 런타임을 유지하세요.")
